# Week 4 Assignment: CIFAR-10 Image Classification using ANN, CNN & Advanced CNN



In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Dropout
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom
from tensorflow.keras.callbacks import EarlyStopping

(x_train,y_train),(x_test,y_test)=cifar10.load_data()

x_train=x_train.astype("float32")/255.0
x_test=x_test.astype("float32")/255.0

class_names=['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

plt.figure(figsize=(10,4))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(x_train[i])
    plt.title(class_names[int(y_train[i])])
    plt.axis("off")
plt.show()

early_stop=EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True)


In [ ]:
ann_model=Sequential([
    Flatten(input_shape=(32,32,3)),
    Dense(512,activation='relu'),
    Dropout(0.3),
    Dense(256,activation='relu'),
    Dropout(0.3),
    Dense(128,activation='relu'),
    Dense(10,activation='softmax')
])

ann_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

ann_history=ann_model.fit(
    x_train,y_train,
    epochs=20,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
cnn_model=Sequential([
    Conv2D(32,(3,3),activation='relu',padding='same',input_shape=(32,32,3)),
    MaxPooling2D(),
    Conv2D(64,(3,3),activation='relu',padding='same'),
    MaxPooling2D(),
    Conv2D(128,(3,3),activation='relu',padding='same'),
    MaxPooling2D(),
    Flatten(),
    Dense(128,activation='relu'),
    Dropout(0.5),
    Dense(10,activation='softmax')
])

cnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

cnn_history=cnn_model.fit(
    x_train,y_train,
    epochs=20,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
data_aug=tf.keras.Sequential([
    RandomFlip('horizontal'),
    RandomRotation(0.1),
    RandomZoom(0.1)
])

aug_model=Sequential([
    data_aug,
    Conv2D(32,(3,3),activation='relu',padding='same',input_shape=(32,32,3)),
    MaxPooling2D(),
    Conv2D(64,(3,3),activation='relu',padding='same'),
    MaxPooling2D(),
    Conv2D(128,(3,3),activation='relu',padding='same'),
    MaxPooling2D(),
    Flatten(),
    Dense(256,activation='relu'),
    Dropout(0.5),
    Dense(10,activation='softmax')
])

aug_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

aug_history=aug_model.fit(
    x_train,y_train,
    epochs=20,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
ann_test=ann_model.evaluate(x_test,y_test,verbose=0)
cnn_test=cnn_model.evaluate(x_test,y_test,verbose=0)
aug_test=aug_model.evaluate(x_test,y_test,verbose=0)

results=pd.DataFrame({
'Model':['ANN','CNN','Advanced CNN'],
'Test Accuracy':[ann_test[1],cnn_test[1],aug_test[1]],
'Test Loss':[ann_test[0],cnn_test[0],aug_test[0]]
})
print(results)

plt.figure(figsize=(8,5))
plt.plot(ann_history.history['val_accuracy'],label='ANN')
plt.plot(cnn_history.history['val_accuracy'],label='CNN')
plt.plot(aug_history.history['val_accuracy'],label='Advanced CNN')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.grid(True)
plt.show()
